# Q1 — Knowledge Distillation Framework (CIFAR-100)

**Complete end-to-end notebook:** ResNet-50 teacher, three student baselines, soft-target + feature-based knowledge distillation, evaluation, ONNX export, and ONNX Runtime CPU inference.



In [6]:
# Cell 1 — Install dependencies
!pip -q install timm onnx onnxruntime

In [7]:
# Cell 2 — Imports, reproducibility, device, and project folders
import os, time, copy, random, warnings, json, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

import timm
import onnx
import onnxruntime as ort

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("timm:", timm.__version__)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Q1_Knowledge_Distillation")
except Exception:
    ROOT = Path("./Q1_Knowledge_Distillation")

DIRS = {
    "data": ROOT / "data",
    "teacher": ROOT / "checkpoints" / "teacher",
    "baseline": ROOT / "checkpoints" / "baseline",
    "distilled": ROOT / "checkpoints" / "distilled",
    "onnx": ROOT / "onnx",
    "results": ROOT / "results",
    "plots": ROOT / "results" / "plots",
    "inference": ROOT / "inference",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

# Main training settings
BATCH_SIZE = 128
NUM_WORKERS = 2
TEACHER_EPOCHS = 30
BASELINE_EPOCHS = 20
KD_EPOCHS = 20
LEARNING_RATE_TEACHER = 0.1
LEARNING_RATE_STUDENT = 0.001
WEIGHT_DECAY = 1e-4

# Reuse existing best checkpoints by default.
FORCE_RETRAIN = False

NUM_CLASSES = 100
CLASS_NAMES = datasets.CIFAR100(root=str(DIRS["data"]), train=True, download=True).classes
print("Classes:", len(CLASS_NAMES))
print("Project folder:", ROOT)

Device: cpu
PyTorch: 2.14.1+cpu
timm: 1.0.31
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Classes: 100
Project folder: /content/drive/MyDrive/Q1_Knowledge_Distillation


In [8]:
# Cell 3 — CIFAR-100 datasets and loaders
CIFAR_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR_STD  = (0.2675, 0.2565, 0.2761)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# Separate dataset objects ensure validation uses deterministic transforms.
full_train_aug = datasets.CIFAR100(
    root=str(DIRS["data"]), train=True, download=True,
    transform=train_transform
)
full_train_eval = datasets.CIFAR100(
    root=str(DIRS["data"]), train=True, download=True,
    transform=eval_transform
)
test_dataset = datasets.CIFAR100(
    root=str(DIRS["data"]), train=False, download=True,
    transform=eval_transform
)

split_generator = torch.Generator().manual_seed(SEED)
train_subset, val_subset = random_split(
    range(len(full_train_aug)), [45000, 5000], generator=split_generator
)

# Subset wrappers use the correct transform for each split.
from torch.utils.data import Subset
train_dataset = Subset(full_train_aug, train_subset.indices)
val_dataset = Subset(full_train_eval, val_subset.indices)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=USE_AMP
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=USE_AMP
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=USE_AMP
)

print("Train:", len(train_dataset), "Validation:", len(val_dataset),
      "Test:", len(test_dataset))

Train: 45000 Validation: 5000 Test: 10000


## Part A — Teacher model: ResNet-50

The teacher is adapted for CIFAR-100's 32×32 images by using a 3×3 first convolution with stride 1 and removing the initial max-pooling layer.

In [9]:
# Cell 4 — ResNet-50 teacher with intermediate features
class ResNet50Teacher(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.backbone = models.resnet50(weights=None)
        self.backbone.conv1 = nn.Conv2d(
            3, 64, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.backbone.maxpool = nn.Identity()
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, num_classes)

    def forward(self, x, return_features=False):
        b = self.backbone
        x = b.conv1(x)
        x = b.bn1(x)
        x = b.relu(x)
        x = b.maxpool(x)
        x = b.layer1(x); f1 = x
        x = b.layer2(x); f2 = x
        x = b.layer3(x); f3 = x
        x = b.layer4(x); f4 = x
        pooled = b.avgpool(x).flatten(1)
        logits = b.fc(pooled)
        if return_features:
            return logits, {"layer1": f1, "layer2": f2,
                            "layer3": f3, "layer4": f4, "pooled": pooled}
        return logits

def make_scaler():
    # Compatible with current Colab PyTorch versions.
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

def evaluate(model, loader, criterion=None):
    model.eval()
    criterion = criterion or nn.CrossEntropyLoss()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(images)
                loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, 100.0 * correct / total

def train_supervised(model, train_loader, val_loader, epochs, lr,
                     optimizer_name="adamw", save_prefix="model"):
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    if optimizer_name == "sgd":
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                              weight_decay=WEIGHT_DECAY)
    else:
        optimizer = optim.AdamW(model.parameters(), lr=lr,
                                weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = make_scaler()
    best_acc = -1.0
    history = []
    if save_prefix == "teacher_resnet50":
        best_path = DIRS["teacher"] / "teacher_resnet50_best.pth"
    elif save_prefix.startswith("baseline_"):
        student_key = save_prefix.replace("baseline_", "")
        best_path = DIRS["baseline"] / f"{student_key}_baseline_best.pth"
    else:
        raise ValueError(f"Unsupported checkpoint prefix: {save_prefix}")
    latest_path = best_path.with_name(best_path.stem.replace("_best", "_latest") + ".pth")

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        start = time.time()
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = 100.0 * correct / total
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        scheduler.step()
        row = {
            "epoch": epoch, "train_loss": train_loss,
            "train_accuracy": train_acc, "val_loss": val_loss,
            "val_accuracy": val_acc,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "epoch_time": time.time() - start
        }
        history.append(row)
        state = {
            "epoch": epoch, "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_accuracy": val_acc, "history": history
        }
        torch.save(state, latest_path)
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(state, best_path)
        pd.DataFrame(history).to_csv(
            DIRS["results"] / f"{save_prefix}_history.csv", index=False
        )
        print(f"Epoch {epoch:02d}/{epochs} | "
              f"train {train_loss:.4f}/{train_acc:.2f}% | "
              f"val {val_loss:.4f}/{val_acc:.2f}% | "
              f"{row['epoch_time']:.1f}s")

    best = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best["model_state_dict"])
    return model, pd.DataFrame(history), best_path

def load_checkpoint_model(model, path):
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    return model.to(DEVICE), checkpoint

In [ ]:
# Cell 5 — Train or load teacher
teacher_path = DIRS["teacher"] / "teacher_resnet50_best.pth"
teacher_model = ResNet50Teacher(NUM_CLASSES).to(DEVICE)

if teacher_path.exists() and not FORCE_RETRAIN:
    teacher_model, teacher_ckpt = load_checkpoint_model(teacher_model, teacher_path)
    print("Loaded existing teacher checkpoint:", teacher_path)
else:
    teacher_model, teacher_history, teacher_path = train_supervised(
        teacher_model, train_loader, val_loader,
        TEACHER_EPOCHS, LEARNING_RATE_TEACHER,
        optimizer_name="sgd", save_prefix="teacher_resnet50"
    )

teacher_test_loss, teacher_test_acc = evaluate(teacher_model, test_loader)
print(f"Teacher test loss: {teacher_test_loss:.4f}")
print(f"Teacher test accuracy: {teacher_test_acc:.2f}%")

## Part B — Student architectures and supervised baselines

All three students output 100 logits. SqueezeNet's final ReLU is explicitly removed: classification logits must be allowed to be negative. This avoids the zero-logit/zero-gradient issue discovered during debugging.

In [ ]:
# Cell 6 — Student architectures
class MobileNetV2Student(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.backbone = models.mobilenet_v2(weights=None, num_classes=num_classes)
        old = self.backbone.features[0][0]
        self.backbone.features[0][0] = nn.Conv2d(
            3, old.out_channels, kernel_size=3, stride=1, padding=1, bias=False
        )

    def forward(self, x, return_features=False):
        fmap = self.backbone.features(x)
        pooled = F.adaptive_avg_pool2d(fmap, 1).flatten(1)
        logits = self.backbone.classifier(pooled)
        if return_features:
            return logits, {"spatial": fmap, "pooled": pooled}
        return logits


class SqueezeNetStudent(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.backbone = models.squeezenet1_1(weights=None)
        self.backbone.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=1)
        # Critical fix: remove ReLU after final classification convolution.
        self.backbone.classifier[2] = nn.Identity()

    def forward(self, x, return_features=False):
        fmap = self.backbone.features(x)
        z = self.backbone.classifier[0](fmap)  # dropout
        z = self.backbone.classifier[1](z)    # raw class scores
        z = self.backbone.classifier[2](z)    # identity, not ReLU
        logits = self.backbone.classifier[3](z).flatten(1)
        pooled = F.adaptive_avg_pool2d(fmap, 1).flatten(1)
        if return_features:
            return logits, {"spatial": fmap, "pooled": pooled}
        return logits


class EfficientNetLite0Student(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        model_name = "tf_efficientnet_lite0"
        available = timm.list_models("*efficientnet*lite*")
        if model_name not in available:
            raise RuntimeError(
                f"{model_name} is unavailable in this timm version. "
                f"Available Lite models: {available}"
            )
        self.backbone = timm.create_model(
            model_name, pretrained=False, num_classes=num_classes, in_chans=3
        )

    def forward(self, x, return_features=False):
        fmap = self.backbone.forward_features(x)
        logits = self.backbone.forward_head(fmap)
        pooled = self.backbone.forward_head(fmap, pre_logits=True)
        if return_features:
            return logits, {"spatial": fmap, "pooled": pooled}
        return logits


student_factories = {
    "MobileNetV2": MobileNetV2Student,
    "SqueezeNet": SqueezeNetStudent,
    "EfficientNet-Lite0": EfficientNetLite0Student,
}
student_keys = {
    "MobileNetV2": "mobilenetv2",
    "SqueezeNet": "squeezenet",
    "EfficientNet-Lite0": "efficientnet_lite0",
}

# Architecture smoke test
for name, factory in student_factories.items():
    model = factory(NUM_CLASSES).to(DEVICE)
    model.eval()
    with torch.no_grad():
        out, feats = model(torch.randn(2, 3, 32, 32, device=DEVICE),
                           return_features=True)
    print(name, "logits:", tuple(out.shape),
          "pooled:", tuple(feats["pooled"].shape),
          "logit std:", round(out.std().item(), 5))
    assert out.shape == (2, NUM_CLASSES)
    assert out.std().item() > 0
    del model

In [ ]:
# Cell 7 — Train or load all three baseline students
baseline_models = {}
baseline_rows = []

for name, factory in student_factories.items():
    key = student_keys[name]
    path = DIRS["baseline"] / f"{key}_baseline_best.pth"
    model = factory(NUM_CLASSES).to(DEVICE)

    if path.exists() and not FORCE_RETRAIN:
        model, ckpt = load_checkpoint_model(model, path)
        print(f"Loaded existing {name} baseline:", path)
    else:
        model, hist, path = train_supervised(
            model, train_loader, val_loader,
            BASELINE_EPOCHS, LEARNING_RATE_STUDENT,
            optimizer_name="adamw", save_prefix=f"baseline_{key}"
        )

    test_loss, test_acc = evaluate(model, test_loader)
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    params = sum(p.numel() for p in model.parameters())
    baseline_models[name] = model
    baseline_rows.append({
        "model": name,
        "training_type": "Baseline",
        "best_validation_accuracy": ckpt.get("val_accuracy", np.nan),
        "test_accuracy": test_acc,
        "test_loss": test_loss,
        "parameters": params
    })
    print(f"{name}: test accuracy={test_acc:.2f}%")

baseline_df = pd.DataFrame(baseline_rows)
baseline_df.to_csv(DIRS["results"] / "baseline_comparison.csv", index=False)
display(baseline_df)

## Part C — Knowledge Distillation

The training objective combines:
- **Hard-label loss:** cross-entropy between student logits and ground-truth labels.
- **Soft-target loss:** temperature-scaled KL divergence between teacher and student predictions.
- **Feature loss:** mean squared error between teacher pooled features and a trainable projection of student pooled features.

The teacher remains frozen. A separate feature projector is trained with each student.

In [ ]:
# Cell 8 — KD helpers and feature projectors
teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad = False

class FeatureProjector(nn.Module):
    def __init__(self, student_dim, teacher_dim=2048):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(student_dim, teacher_dim),
            nn.ReLU(inplace=True),
            nn.Linear(teacher_dim, teacher_dim)
        )

    def forward(self, x):
        return self.proj(x)

def infer_feature_dims(student):
    student.eval()
    with torch.no_grad():
        _, sf = student(
            torch.zeros(2, 3, 32, 32, device=DEVICE),
            return_features=True
        )
        _, tf = teacher_model(
            torch.zeros(2, 3, 32, 32, device=DEVICE),
            return_features=True
        )
    return sf["pooled"].shape[1], tf["pooled"].shape[1]

def kd_loss_function(student_logits, teacher_logits, labels,
                     student_features, teacher_features, projector,
                     temperature=4.0, alpha=0.7, beta=0.1):
    hard_loss = F.cross_entropy(student_logits, labels)
    soft_loss = F.kl_div(
        F.log_softmax(student_logits / temperature, dim=1),
        F.softmax(teacher_logits / temperature, dim=1),
        reduction="batchmean"
    ) * (temperature ** 2)
    projected = projector(student_features)
    feature_loss = F.mse_loss(projected, teacher_features)
    total = (1 - alpha) * hard_loss + alpha * soft_loss + beta * feature_loss
    return total, hard_loss.detach(), soft_loss.detach(), feature_loss.detach()

In [ ]:
# Cell 9 — Train or load distilled students
TEMPERATURE = 4.0
ALPHA = 0.7
BETA = 0.1

distilled_models = {}
projectors = {}
kd_rows = []

for name, factory in student_factories.items():
    key = student_keys[name]
    student_path = DIRS["distilled"] / f"{key}_kd_best.pth"
    projector_path = DIRS["distilled"] / f"{key}_projector_best.pth"

    student = factory(NUM_CLASSES).to(DEVICE)
    student_dim, teacher_dim = infer_feature_dims(student)
    projector = FeatureProjector(student_dim, teacher_dim).to(DEVICE)

    if student_path.exists() and projector_path.exists() and not FORCE_RETRAIN:
        student_ckpt = torch.load(student_path, map_location=DEVICE, weights_only=False)
        proj_ckpt = torch.load(projector_path, map_location=DEVICE, weights_only=False)
        student.load_state_dict(student_ckpt["model_state_dict"])
        projector.load_state_dict(proj_ckpt["model_state_dict"])
        print(f"Loaded existing KD model: {name}")
    else:
        optimizer = optim.AdamW(
            list(student.parameters()) + list(projector.parameters()),
            lr=LEARNING_RATE_STUDENT, weight_decay=WEIGHT_DECAY
        )
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=KD_EPOCHS
        )
        scaler = make_scaler()
        best_val = -1.0
        history = []

        for epoch in range(1, KD_EPOCHS + 1):
            student.train()
            projector.train()
            total_loss = 0.0
            correct = 0
            total = 0
            start = time.time()

            for images, labels in train_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)

                with torch.no_grad():
                    with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                        teacher_logits, teacher_feats = teacher_model(
                            images, return_features=True
                        )

                with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    student_logits, student_feats = student(
                        images, return_features=True
                    )
                    loss, hard_l, soft_l, feat_l = kd_loss_function(
                        student_logits, teacher_logits, labels,
                        student_feats["pooled"], teacher_feats["pooled"],
                        projector, TEMPERATURE, ALPHA, BETA
                    )

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                total_loss += loss.item() * labels.size(0)
                correct += (student_logits.argmax(1) == labels).sum().item()
                total += labels.size(0)

            train_loss = total_loss / total
            train_acc = 100.0 * correct / total
            val_loss, val_acc = evaluate(student, val_loader)
            scheduler.step()

            row = {
                "epoch": epoch, "train_loss": train_loss,
                "train_accuracy": train_acc, "val_loss": val_loss,
                "val_accuracy": val_acc,
                "learning_rate": optimizer.param_groups[0]["lr"],
                "epoch_time": time.time() - start
            }
            history.append(row)

            student_state = {
                "epoch": epoch, "model_state_dict": student.state_dict(),
                "val_accuracy": val_acc, "history": history
            }
            projector_state = {"model_state_dict": projector.state_dict()}
            torch.save(student_state,
                       DIRS["distilled"] / f"{key}_kd_latest.pth")
            torch.save(projector_state,
                       DIRS["distilled"] / f"{key}_projector_latest.pth")

            if val_acc > best_val:
                best_val = val_acc
                torch.save(student_state, student_path)
                torch.save(projector_state, projector_path)

            pd.DataFrame(history).to_csv(
                DIRS["results"] / f"{key}_kd_history.csv", index=False
            )
            print(f"{name} | Epoch {epoch:02d}/{KD_EPOCHS} | "
                  f"train {train_loss:.4f}/{train_acc:.2f}% | "
                  f"val {val_loss:.4f}/{val_acc:.2f}%")

        best_student = torch.load(student_path, map_location=DEVICE, weights_only=False)
        best_projector = torch.load(projector_path, map_location=DEVICE, weights_only=False)
        student.load_state_dict(best_student["model_state_dict"])
        projector.load_state_dict(best_projector["model_state_dict"])

    student.eval()
    projector.eval()
    test_loss, test_acc = evaluate(student, test_loader)
    params = sum(p.numel() for p in student.parameters())
    distilled_models[name] = student
    projectors[name] = projector
    kd_rows.append({
        "model": name,
        "training_type": "Knowledge Distillation",
        "best_validation_accuracy": float(torch.load(
            student_path, map_location="cpu", weights_only=False
        ).get("val_accuracy", np.nan)),
        "test_accuracy": test_acc,
        "test_loss": test_loss,
        "parameters": params
    })
    print(f"{name} KD test accuracy: {test_acc:.2f}%")

kd_df = pd.DataFrame(kd_rows)
kd_df.to_csv(DIRS["results"] / "distilled_comparison.csv", index=False)
display(kd_df)

In [ ]:
# Cell 10 — Combined baseline vs KD comparison
comparison_df = pd.concat([baseline_df, kd_df], ignore_index=True)
comparison_df.to_csv(DIRS["results"] / "baseline_vs_kd_comparison.csv", index=False)
display(comparison_df)

pivot = comparison_df.pivot(
    index="model", columns="training_type", values="test_accuracy"
)
pivot["accuracy_change_after_kd"] = (
    pivot["Knowledge Distillation"] - pivot["Baseline"]
)
display(pivot)
pivot.to_csv(DIRS["results"] / "baseline_vs_kd_accuracy.csv")

## Part D — Unified inference

A single helper accepts a student model name and a batch of normalized CIFAR-100 images.

In [ ]:
# Cell 11 — Unified inference helper
ALL_STUDENT_MODELS = {
    **{f"{name} (Baseline)": model for name, model in baseline_models.items()},
    **{f"{name} (KD)": model for name, model in distilled_models.items()},
}

@torch.no_grad()
def predict(model_name, images):
    if model_name not in ALL_STUDENT_MODELS:
        raise ValueError(f"Unknown model. Choose from: {list(ALL_STUDENT_MODELS)}")
    model = ALL_STUDENT_MODELS[model_name].to(DEVICE).eval()
    if images.ndim == 3:
        images = images.unsqueeze(0)
    logits = model(images.to(DEVICE))
    probs = torch.softmax(logits, dim=1)
    conf, pred = probs.max(dim=1)
    return [
        {"class_id": int(p), "class_name": CLASS_NAMES[int(p)],
         "confidence": float(c)}
        for p, c in zip(pred.cpu(), conf.cpu())
    ]

sample_images, sample_labels = next(iter(test_loader))
print("Actual:", [CLASS_NAMES[int(x)] for x in sample_labels[:5]])
print("Predictions:", predict("MobileNetV2 (KD)", sample_images[:5]))

## Part E — ONNX export and CPU inference

Each student (baseline and distilled) is exported to ONNX. ONNX Runtime uses the CPU execution provider for inference and benchmarking.

In [ ]:
# Cell 12 — Export all student models to ONNX
class LogitsOnly(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        return self.model(x)

onnx_models = {}
dummy = torch.randn(1, 3, 32, 32, device=DEVICE)

for label, model in ALL_STUDENT_MODELS.items():
    safe_name = (
        label.lower().replace(" ", "_")
        .replace("(", "").replace(")", "")
    )
    output_path = DIRS["onnx"] / f"{safe_name}.onnx"
    wrapper = LogitsOnly(model.eval()).to(DEVICE)

    torch.onnx.export(
        wrapper, dummy, str(output_path),
        input_names=["input"], output_names=["logits"],
        dynamic_axes={
            "input": {0: "batch_size"},
            "logits": {0: "batch_size"}
        },
        opset_version=17,
        do_constant_folding=True
    )
    onnx.checker.check_model(onnx.load(str(output_path)))
    onnx_models[label] = output_path
    print("Exported:", output_path)

In [ ]:
# Cell 13 — ONNX Runtime CPU inference and benchmark
def benchmark_onnx(onnx_path, loader, max_batches=None):
    session = ort.InferenceSession(
        str(onnx_path),
        providers=["CPUExecutionProvider"]
    )
    input_name = session.get_inputs()[0].name
    total_images = 0
    correct = 0
    total_loss = 0.0
    elapsed = 0.0

    for batch_index, (images, labels) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        x = images.numpy().astype(np.float32)
        y = labels.numpy()
        start = time.perf_counter()
        logits = session.run(None, {input_name: x})[0]
        elapsed += time.perf_counter() - start

        logits_t = torch.from_numpy(logits)
        labels_t = torch.from_numpy(y)
        total_loss += F.cross_entropy(logits_t, labels_t, reduction="sum").item()
        correct += (logits.argmax(axis=1) == y).sum()
        total_images += len(y)

    return {
        "onnx_test_loss": total_loss / total_images,
        "onnx_test_accuracy": 100.0 * correct / total_images,
        "total_inference_time_seconds": elapsed,
        "average_latency_ms_per_image": elapsed / total_images * 1000,
        "throughput_images_per_second": total_images / elapsed
    }

onnx_rows = []
for label, path in onnx_models.items():
    metrics = benchmark_onnx(path, test_loader)
    size_mb = os.path.getsize(path) / (1024 ** 2)
    row = {"model": label, "model_size_MB": size_mb, **metrics}
    onnx_rows.append(row)
    print(label, row)

onnx_df = pd.DataFrame(onnx_rows)
onnx_df.to_csv(DIRS["results"] / "onnx_cpu_benchmark.csv", index=False)
display(onnx_df)

In [ ]:
# Cell 14 — Validate PyTorch vs ONNX output for one batch
validation_rows = []
images, labels = next(iter(test_loader))
x_np = images[:8].numpy().astype(np.float32)

for label, path in onnx_models.items():
    model = ALL_STUDENT_MODELS[label].eval()
    with torch.no_grad():
        torch_logits = model(images[:8].to(DEVICE)).cpu().numpy()

    session = ort.InferenceSession(
        str(path), providers=["CPUExecutionProvider"]
    )
    onnx_logits = session.run(
        None, {session.get_inputs()[0].name: x_np}
    )[0]

    max_diff = float(np.max(np.abs(torch_logits - onnx_logits)))
    validation_rows.append({"model": label, "max_absolute_difference": max_diff})
    print(label, "max abs difference:", max_diff)

onnx_validation_df = pd.DataFrame(validation_rows)
display(onnx_validation_df)
onnx_validation_df.to_csv(
    DIRS["results"] / "onnx_validation.csv", index=False
)

## Part F — Final results and deliverables

All generated artifacts are stored in the project folder in Google Drive:
- `checkpoints/teacher/`
- `checkpoints/baseline/`
- `checkpoints/distilled/`
- `onnx/`
- `results/`

This final cell prints the available files and saves a JSON summary.

In [ ]:
# Cell 15 — Save final Q1 summary
summary = {
    "dataset": "CIFAR-100",
    "num_classes": NUM_CLASSES,
    "teacher": {
        "architecture": "ResNet-50",
        "test_accuracy": teacher_test_acc,
        "test_loss": teacher_test_loss
    },
    "baseline_results": baseline_df.to_dict(orient="records"),
    "distilled_results": kd_df.to_dict(orient="records"),
    "onnx_cpu_results": onnx_df.to_dict(orient="records")
}

summary_path = DIRS["results"] / "q1_final_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Q1 pipeline completed.")
print("Summary:", summary_path)
print("\nProject output folders:")
for key, path in DIRS.items():
    print(f"{key:12s}: {path}")